# Writing files out

Everything so far has been read-only, but a real pipeline produces something and writing is
often the slowest part.

* **O1** read, transform, write one Parquet file
* **O2** the same, partitioned by month
* **O3** compaction, 200 small files in and one out
* **O4** CSV to Parquet

These are the only cases needing different code per engine, because writing is expressed
differently in each. The `SELECT` is still identical.

On Windows, Spark needs `winutils.exe` to write files. Without it these four fail and the rest
of the project carries on.

In [ ]:
import sys, os, shutil, glob
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "08_write_output", C.MAIN_SIZE)

OUT = C.OUT_DIR.replace("\\", "/")
def fresh(p):
    if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True)
    elif os.path.exists(p): os.remove(p)
    return p

print(f"Ready. Output goes to {OUT}")


In [ ]:
# O1: read, transform, write one Parquet file
TRANSFORM = """
SELECT sale_id, customer_id, product_id, sale_date, channel, region, quantity, amount,
       round(COALESCE(amount,0) * (1 - discount_pct) * (1 + tax_pct), 2) AS net_amount,
       CASE WHEN amount > 200 THEN 'high' WHEN amount > 50 THEN 'medium' ELSE 'low' END AS band
FROM sales
WHERE amount IS NOT NULL AND amount > 0 AND channel IN ('web','app')
"""

def duck_o1():
    p = fresh(f"{OUT}/duck_single.parquet")
    duck.execute(f"COPY ({TRANSFORM}) TO '{p}' (FORMAT PARQUET)")
    return duck.execute(f"SELECT count(*) AS n FROM read_parquet('{p}')").df()

def spark_o1():
    p = fresh(f"{OUT}/spark_single")
    spark.sql(TRANSFORM).write.mode("overwrite").parquet(p)
    import pandas as pd
    return pd.DataFrame({"n": [spark.read.parquet(p).count()]})

_, out, _ = bench.run(Case("O1", "Transform and write one file", "Write",
                           duck_fn=duck_o1, spark_fn=spark_o1 if spark else None))
display(out)


In [ ]:
# O2: write partitioned by month
def duck_o2():
    p = fresh(f"{OUT}/duck_part")
    duck.execute(f"""COPY (
        SELECT sale_id, customer_id, amount, channel,
               CAST(date_trunc('month', sale_date) AS DATE) AS sale_month
        FROM sales WHERE amount IS NOT NULL
    ) TO '{p}' (FORMAT PARQUET, PARTITION_BY (sale_month), OVERWRITE_OR_IGNORE 1)""")
    return duck.execute(
        f"SELECT count(*) AS n FROM read_parquet('{p}/**/*.parquet')").df()

def spark_o2():
    p = fresh(f"{OUT}/spark_part")
    (spark.sql("""
        SELECT sale_id, customer_id, amount, channel,
               CAST(date_trunc('month', sale_date) AS DATE) AS sale_month
        FROM sales WHERE amount IS NOT NULL
     """).write.mode("overwrite").partitionBy("sale_month").parquet(p))
    import pandas as pd
    return pd.DataFrame({"n": [spark.read.parquet(p).count()]})

_, out, _ = bench.run(Case("O2", "Write partitioned by month", "Write",
                           duck_fn=duck_o2, spark_fn=spark_o2 if spark else None))
display(out)


In [ ]:
# O3: compaction -- 200 small files in, few big ones out
SMALL = paths["sales_small_files"].replace("\\", "/")
n_small = len(glob.glob(f"{SMALL}/**/*.parquet", recursive=True))
print(f"Input: {n_small} small files\n")

def duck_o3():
    p = fresh(f"{OUT}/duck_compact.parquet")
    duck.execute(f"COPY (SELECT * FROM read_parquet('{SMALL}/**/*.parquet')) "
                 f"TO '{p}' (FORMAT PARQUET)")
    return duck.execute(f"SELECT count(*) AS n FROM read_parquet('{p}')").df()

def spark_o3():
    p = fresh(f"{OUT}/spark_compact")
    spark.read.parquet(SMALL).coalesce(1).write.mode("overwrite").parquet(p)
    import pandas as pd
    return pd.DataFrame({"n": [spark.read.parquet(p).count()]})

_, out, _ = bench.run(Case("O3", f"Compact {n_small} small files into one", "Write",
                           duck_fn=duck_o3, spark_fn=spark_o3 if spark else None))
display(out)


In [ ]:
# O4: CSV to Parquet
CSV = paths["sales_csv"].replace("\\", "/")

def duck_o4():
    p = fresh(f"{OUT}/duck_from_csv.parquet")
    duck.execute(f"COPY (SELECT * FROM read_csv('{CSV}', header=true)) "
                 f"TO '{p}' (FORMAT PARQUET)")
    return duck.execute(f"SELECT count(*) AS n FROM read_parquet('{p}')").df()

def spark_o4():
    p = fresh(f"{OUT}/spark_from_csv")
    (spark.read.option("header", "true").option("inferSchema", "true").csv(CSV)
          .write.mode("overwrite").parquet(p))
    import pandas as pd
    return pd.DataFrame({"n": [spark.read.parquet(p).count()]})

_, out, _ = bench.run(Case("O4", "Convert CSV to Parquet", "Write",
                           duck_fn=duck_o4, spark_fn=spark_o4 if spark else None))
display(out)


## Results for this notebook

All four of these needed engine-specific code, because writing files is expressed differently
in each engine. That is worth being straight about: the *reading and transforming* transfers
unchanged between them, the *writing* does not.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()
